In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [2]:
from aiida import orm
from monty.serialization import loadfn, dumpfn
import numpy as np
from aiida_vasp.workchains import VaspHybridBandsWorkChain
from aiida_vasp.workchains.v2 import VaspHybridBandUpdater, VaspRelaxUpdater
from aiida_grouppathx import GroupPathX
from tqdm import tqdm

In [3]:
basepath = GroupPathX('hc-ternary-mbj')
structpath = basepath['structures']

Deposit the structures

Sort the nodes based on their number of sites

## Setting up the calculation

In [4]:
def get_upd(node):
    """Generate process builder"""
    upd = VaspHybridBandUpdater().apply_preset(structure=node, overrides={
        'metagga': 'mbj',
        'ispin': 2,
        'gga': None,
        'magmom': None,
        'ncore': 8,
    },
         code='vasp-6.3.0@sugon-xian-v2', label=f'{node.get_formula()} {node.label} MP STRUCT MBJ NORELAX')
    upd.set_resources(num_machines=1, tot_num_mpiprocs=64)
    upd.set_options(max_wallclock_seconds=3600 * 48, queue_name='xahcnormal')
    # Reuse WAVECAR for the mbj band structure calculation
    upd.set_band_settings(hybrid_reuse_wavecar=True)
    
    # Only re-try twice if there is convergence problem
    upd.builder.scf.max_iterations = 2
    
    upd_relax = VaspRelaxUpdater(builder=upd.builder.relax)
    upd_relax.apply_preset(structure=node, overrides={
        #'metagga': 'mbj',
        'ispin': 2,
        'gga': None,
        'magmom': None,
        'ncore': 8,
    },
         code='vasp-6.3.0-autotruncate@sugon-xian-v2', label=f'{node.get_formula()} {node.label} MP STRUCT SP')
    upd_relax.set_resources(num_machines=1, tot_num_mpiprocs=64)
    upd_relax.set_options(max_wallclock_seconds=3600 * 12, queue_name='xahcnormal')
    upd_relax.set_relax_settings(perform=False)
    upd.set_band_settings(band_mode='bradcrack', line_density=20, kpoints_per_split=200)
    return upd

In [5]:
workpath = basepath['bandstructure_works']

## Fix the excepted jobs

In [6]:
errored = []
for path in workpath:
    node = path.get_node()
    if not node.is_finished_ok:
        errored.append([path.key, node])
        print(node.label, node.pk, node.exit_status, node.process_state)

BiLuNi mp-30457 MP STRUCT MBJ NORELAX 99568 None ProcessState.EXCEPTED
BiGdPd mp-1076916 MP STRUCT MBJ NORELAX 99831 None ProcessState.EXCEPTED
CeNiSb mp-1226516 MP STRUCT MBJ NORELAX 99955 501 ProcessState.FINISHED
BiLaPd mp-1206717 MP STRUCT MBJ NORELAX 100275 None ProcessState.EXCEPTED
GdNiSb mp-569197 MP STRUCT MBJ NORELAX 100634 None ProcessState.EXCEPTED
BiLuPd mp-1207185 MP STRUCT MBJ NORELAX 100746 None ProcessState.EXCEPTED
BiGdNi mp-567422 MP STRUCT MBJ NORELAX 100857 None ProcessState.EXCEPTED
BiLaPt mp-1018136 MP STRUCT MBJ NORELAX 101057 None ProcessState.EXCEPTED
BiErPd mp-1206953 MP STRUCT MBJ NORELAX 101124 None ProcessState.EXCEPTED
LaLi2Sn mp-861894 MP STRUCT MBJ NORELAX 101279 None ProcessState.EXCEPTED
HfLuRu2 mp-865846 MP STRUCT MBJ NORELAX 101369 501 ProcessState.FINISHED
C2CoEr mp-13501 MP STRUCT MBJ NORELAX 101435 None ProcessState.EXCEPTED
CeNdS2 mp-1226502 MP STRUCT MBJ NORELAX 103845 None ProcessState.EXCEPTED
LaN2Nd mp-1222795 MP STRUCT MBJ NORELAX 103868 No

In [7]:
rerun_path = basepath['bandstructure_errored_rerun']
for name, node in errored[:1]:
    upd = get_upd(node.inputs.structure)
    upd.set_incar(algo='all')
    running = upd.submit()
    rerun_path.add_node(running, name, True)

/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_grouppathx/pathx.py:385: UserWarning: Alias mp-30457 is not valid Python identifier - you may consider using one for easier access.
  warnings.warn(


In [72]:
for path in basepath['bandstructure_excepted_rerun']:
    node = path.get_node()
    print(path.key, node.is_finished_ok, node.is_excepted, node.pk)

mp-30457 False True 189164
mp-1076916 False True 189167
mp-1206717 False True 189175
mp-1207185 False True 189184
mp-567422 False True 189186
mp-1018136 False True 189190
mp-861894 False True 189200
mp-13501 False True 189207
mp-569197 False True 189420
mp-1206953 False True 189423
mp-862778 True False 189431
mp-1183483 True False 189438
mp-865680 True False 189440
mp-5149 True False 189442
mp-1185329 True False 189446
mp-865681 True False 189449
mp-996994 True False 189462
mp-21883 False True 189471
mp-861976 False True 189716
mp-4226 False True 189718
mp-862288 True False 189728
mp-866193 False True 189736
mp-865010 True False 189738
mp-1224247 True False 189739
mp-865092 True False 189742
mp-864791 False True 189748
mp-20584 False True 189752
mp-12129 False True 189756
mp-867120 True False 189759
mp-568880 True False 189764
mp-1222918 False True 189774
mp-1226534 False True 189781
mp-5778 False True 189794
mp-865884 False True 189801
mp-865081 True False 189804
mp-1221073 False True

In [68]:
to_retry = []
for path in basepath['bandstructure_excepted_rerun']:
    node = path.get_node()
    if not node.is_finished_ok:
        if all(x.is_finished_ok for x in node.called_descendants):
            to_retry.append(path)